# Notebook 1: Attention, Live

**Goal:** Get hands-on with scaled dot-product / multi-head attention as fast as possible. The implementation below is already written for you — don't worry about building it from scratch right now. Run the setup cells, then jump straight to the **tweaking section** and start changing things.

**Time box: ~20 minutes.**


## Setup (run these cells, don't edit yet)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)


In [ ]:
def scaled_dot_product_attention(Q, K, V, use_scaling=True, use_softmax=True, causal_mask=False):
    """
    Q, K, V: tensors of shape (num_heads, seq_len, d_k)
    Returns: output (num_heads, seq_len, d_k), attn_weights (num_heads, seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1)  # (num_heads, seq_len, seq_len)

    if use_scaling:
        scores = scores / np.sqrt(d_k)

    if causal_mask:
        seq_len = scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

    if use_softmax:
        attn_weights = F.softmax(scores, dim=-1)
    else:
        attn_weights = scores  # raw dot products, no normalization

    output = attn_weights @ V
    return output, attn_weights


def multi_head_split(x, num_heads):
    """x: (seq_len, d_model) -> (num_heads, seq_len, d_k)"""
    seq_len, d_model = x.shape
    d_k = d_model // num_heads
    return x.view(seq_len, num_heads, d_k).transpose(0, 1)


In [ ]:
def plot_attention(attn_weights, tokens, title="Attention weights"):
    """attn_weights: (num_heads, seq_len, seq_len)"""
    num_heads = attn_weights.shape[0]
    fig, axes = plt.subplots(1, num_heads, figsize=(4 * num_heads, 4))
    if num_heads == 1:
        axes = [axes]
    for h in range(num_heads):
        ax = axes[h]
        im = ax.imshow(attn_weights[h].detach().numpy(), cmap="viridis")
        ax.set_xticks(range(len(tokens)))
        ax.set_yticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, ha="right")
        ax.set_yticklabels(tokens)
        ax.set_title(f"Head {h}")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# A fixed toy sentence with random embeddings (untrained -- we're only testing the mechanism)
tokens = ["the", "cat", "sat", "on", "the", "mat"]
seq_len = len(tokens)
d_model = 16

x = torch.randn(seq_len, d_model)  # random "embeddings" standing in for real ones
print(f"{seq_len} tokens, d_model={d_model}")
print(tokens)


## Baseline: run attention as designed

This is the "normal" configuration: scaling on, softmax on, 1 head, no mask. Run it and look at the heatmap — with random untrained embeddings, expect it to look roughly uniform/noisy. That's expected and worth noting.

In [ ]:
NUM_HEADS = 1
Q = multi_head_split(x, NUM_HEADS)
K = multi_head_split(x, NUM_HEADS)
V = multi_head_split(x, NUM_HEADS)

output, attn_weights = scaled_dot_product_attention(Q, K, V, use_scaling=True, use_softmax=True, causal_mask=False)
plot_attention(attn_weights, tokens, title="Baseline: 1 head, scaled, softmax, no mask")


## Now tweak it yourself

Change the flags below one at a time, re-run the cell, and look at how the heatmap changes. Predict what will happen *before* you run it.

Things to try (one at a time):
- `USE_SCALING = False` — remove the `1/sqrt(d_k)` factor
- `USE_SOFTMAX = False` — use raw dot products instead of a normalized distribution
- `NUM_HEADS = 4` — split into more heads (try 2, 4, 8 — must divide d_model=16 evenly)
- `CAUSAL_MASK = True` — mask out attention to future tokens

Try combining a few, too.

In [ ]:
# ---- EDIT THESE ----
USE_SCALING = True
USE_SOFTMAX = True
NUM_HEADS = 1
CAUSAL_MASK = False
# ---------------------

Q = multi_head_split(x, NUM_HEADS)
K = multi_head_split(x, NUM_HEADS)
V = multi_head_split(x, NUM_HEADS)

output, attn_weights = scaled_dot_product_attention(
    Q, K, V,
    use_scaling=USE_SCALING,
    use_softmax=USE_SOFTMAX,
    causal_mask=CAUSAL_MASK,
)
title = f"heads={NUM_HEADS}, scaling={USE_SCALING}, softmax={USE_SOFTMAX}, causal={CAUSAL_MASK}"
plot_attention(attn_weights, tokens, title=title)


### Discussion checkpoints

- What happened to the heatmap when you turned off softmax? Why does that make the output harder to interpret as a "weighted average"?
- What happened without scaling, especially as you increase `d_model` or use more extreme random values? (Hint: think about what large dot products do to softmax.)
- With the causal mask on, which part of the heatmap became zero, and why does that matter for a model that generates text left-to-right?
- Do different heads (with `NUM_HEADS > 1`) end up looking different from each other, even though the embeddings are random and untrained? What does that tell you about how much of "attention" is architecture vs. learned weights?
